# Movement classification with Nearest Centroid

In [9]:
import numpy as np
import pandas as pd

## Load the data

In [10]:
data = pd.read_csv(
    '/Users/blitz/Documents/aiMinorChallenge/REHAB/Rehab_exercise/d03_feature_data/rehab_exercise_features.csv'
)

feature_columns = data.columns[8:]
data.head()

,record_id,source_sample_id,movement_id,movement_type,window_id,start_timepoint,end_timepoint,split,pitch_1_std,pitch_1_median,...,finger_5_min,finger_5_max,finger_5_iqr,finger_5_mad_diff,wrist_pitch_std,wrist_pitch_median,wrist_pitch_min,wrist_pitch_max,wrist_pitch_iqr,wrist_pitch_mad_diff
0,0,0,0,bobath_handshake,0,0,109,train,27.407620,-14.465941,...,-9.145568,6.354432,9.825,0.259633,69.144078,43.791327,-95.283864,85.718660,135.985946,2.531138
1,0,0,0,bobath_handshake,1,110,219,train,32.124926,28.154059,...,-5.945568,5.254432,1.000,0.141284,70.721517,-78.907298,-91.496040,102.666751,91.967748,2.549152
2,0,0,0,bobath_handshake,2,220,329,train,19.543044,-31.069241,...,-6.945568,4.254432,3.275,0.311009,68.491422,79.377449,-81.726537,112.044924,113.431176,3.701181
3,0,0,0,bobath_handshake,3,330,439,train,16.822138,29.793209,...,-2.745568,0.054432,1.450,0.092661,75.773251,-51.502727,-82.993920,111.134494,156.478068,2.085267
4,0,0,0,bobath_handshake,4,440,549,train,15.094130,-39.042191,...,-6.545568,4.254432,4.275,0.211009,80.641894,-73.154802,-83.343424,113.818229,166.163633,2.149617


## Prepare the training and test data

In [11]:
train_data = data[data['split'] == 'train']
test_data = data[data['split'] == 'test']

X_train = train_data[feature_columns].to_numpy()
y_train = train_data['movement_id'].to_numpy()

X_test = test_data[feature_columns].to_numpy()
y_test = test_data['movement_id'].to_numpy()

print('Training examples:', len(X_train))
print('Test examples:', len(X_test))

Training examples: 29544
Test examples: 7384


## Put the features on a similar scale

In [12]:
mean = X_train.mean(axis=0)
standard_deviation = X_train.std(axis=0)

X_train = (X_train - mean) / (standard_deviation + 0.000001)
X_test = (X_test - mean) / (standard_deviation + 0.000001)

## Train the model

In [13]:
centroids = []

for movement_id in range(16):
    movement_examples = X_train[y_train == movement_id]
    centroid = movement_examples.mean(axis=0)
    centroids.append(centroid)

centroids = np.array(centroids)
print('Centroids created:', len(centroids))

Centroids created: 16


## Make predictions

In [14]:
predictions = []

for example in X_test:
    distances = []

    for centroid in centroids:
        distance = np.sqrt(np.sum((example - centroid) ** 2))
        distances.append(distance)

    predicted_movement = np.argmin(distances)
    predictions.append(predicted_movement)

predictions = np.array(predictions)

## Evaluate the result

In [15]:
correct_predictions = predictions == y_test
accuracy = correct_predictions.mean()

print('Accuracy:', round(accuracy * 100, 2), '%')

Accuracy: 48.79 %
